In [2]:
# Step 1: Data Collection

import pandas as pd
from google.colab import files

uploaded = files.upload()

df =pd.read_csv("knowledge_base.csv")


Saving knowledge_base.csv to knowledge_base.csv


In [3]:
print(df.head())

print("\nDataSet Shape:",df.shape)

print("\nColumns:",df.columns.tolist())

  document_id        category  \
0    KB000001  Authentication   
1    KB000002  Authentication   
2    KB000003  Authentication   
3    KB000004  Authentication   
4    KB000005  Authentication   

                                               title  \
0  Unable to Access Your Login Password Step by Step   
1                 Login Troubleshooting the Easy Way   
2                        Setting Up Extra Login Step   
3        Verification Code Troubleshooting Explained   
4  How to Set Up Your Authentication App the Easy...   

                                             content  \
0  If you can no longer use your sign-in password...   
1  Problems with your log in are often caused by ...   
2  You can set up your two-step verification your...   
3  Most issues with your login code clear up afte...   
4  Setting up your authentication app takes only ...   

                                            keywords  
0  password recovery, reset password, identity ve...  
1  login not work

In [4]:
# ==========================================
# STEP 2: DATA UNDERSTANDING
# ==========================================

# 1. Dataset information
print("DATASET INFROMATION")
df.info()

#2.Missing Values
print("\nMissing Values:")
print(df.isnull().sum())

#3.Duplicates
print("\nDUPLICATES")
print("Duplicate Rows:",df.duplicated().sum())
print("Duplicated Documents Ids:",df["document_id"].duplicated().sum())

#4.Unique Values
print("\nUNIQUE VALUES:")
print(df.nunique())

#5.Category Distribution
print("\nNUMBER OF CATEGORIES")
print(df["category"].value_counts())

#6.Random Sample
print("\nRANDOM SAMPLE")
display(df.sample(5,random_state=42))

#7.Document Lengths
df["content_word_count"] = df["content"].fillna("").str.split().str.len()

print("\nCONTENT WORD COUNT STATISTICS")
print(df["content_word_count"].describe())

DATASET INFROMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   document_id  50000 non-null  object
 1   category     50000 non-null  object
 2   title        50000 non-null  object
 3   content      50000 non-null  object
 4   keywords     50000 non-null  object
dtypes: object(5)
memory usage: 1.9+ MB

Missing Values:
document_id    0
category       0
title          0
content        0
keywords       0
dtype: int64

DUPLICATES
Duplicate Rows: 0
Duplicated Documents Ids: 0

UNIQUE VALUES:
document_id    50000
category          15
title          50000
content        50000
keywords       49544
dtype: int64

NUMBER OF CATEGORIES
category
Account Management    4143
Payments              3976
Authentication        3941
Orders                3839
Shipping              3690
General FAQ           3419
Technical Support     3250
Customer Support  

,document_id,category,title,content,keywords
33553,KB033554,General FAQ,Viewing Your Warranty on iOS,Knowing how to read your warranty cover helps ...,"check warranty, warranty status, view warranty..."
9427,KB009428,Orders,Troubleshoot Your Confirmation Email Step by Step,Problems with your order receipt are often cau...,"order confirmation not working, order confirma..."
199,KB000200,Authentication,Edit Your Login Email,You can change your account email at any time ...,"update login email, change login email, sign-i..."
12447,KB012448,Shipping,How to Set Up Your Express Delivery,Setting up your priority delivery takes only a...,"set up express delivery, enable express delive..."
39489,KB039490,Shipping,Broken On Arrival Stopped Working,Most issues with your broken on arrival clear ...,"damaged delivery not working, damaged delivery..."



CONTENT WORD COUNT STATISTICS
count    50000.000000
mean       151.863420
std         18.891453
min        123.000000
25%        137.000000
50%        148.000000
75%        164.000000
max        234.000000
Name: content_word_count, dtype: float64


In [5]:
# ==========================================
# STEP 3: DATA CLEANING
# ==========================================

import re

#1.Combining usefule text columns

df["search_text"]=(df["title"].fillna("")+" "+
                   df["content"].fillna(" ")+" "+
                   df["keywords"].fillna(""))

#2.Text Cleaning Function

def clean_text(text):
  #Convert text to lowercase
  text = text.lower()

  #Remove URLs
  text = re.sub(r"http://S+|www\S+"," ",text)

  #Remove  Punctuation and special characters
  text=re.sub(r"[^a-zA-Z0-9\s]"," ",text)

  #Remove Extra whitespaces
  text=re.sub(r"\s+"," ",text).strip()

  return text

#3.Apply Cleaning

df["clean_text"] = df["search_text"].apply(clean_text)

#4.Check Results

print("Original:")
print(df.loc[0,"search_text"])

print("\nCleaned:")
print(df.loc[0,"clean_text"])

#5.Check Empty documents

print("\nEmpty documents after cleaning:",(df["clean_text"].str.len()==0).sum())

Original:
Unable to Access Your Login Password Step by Step If you can no longer use your sign-in password, you can restore access yourself in a few steps. Go to the sign-in page and select the option to reset your password. Enter the email address linked to your account so a recovery message can be sent. Choose a new password you have not used before, mixing letters, numbers, and symbols. Confirm the new password and sign in again to make sure it works. If you no longer have the linked email, use another method such as a phone number or backup code. After several failed attempts, access may lock briefly, so wait a moment before trying again. Once you are back in, review your recent sign-in activity for anything you do not recognise. Keeping a recovery email and phone number on file makes future resets faster. A password manager can store your new password so you do not lose it again. If these steps do not resolve it, contact support with your account email and any error message so we 

In [6]:
# ==========================================
# STEP 3 & 4: TEXT CLEANING + PREPROCESSING
# ==========================================

import re
import nltk
from nltk.corpus import stopwords

#Download NLTK Stopwords
nltk.download("stopwords")

#Creating stopword list

stop_words=set(stopwords.words("english"))

#Keeping important Negation Words

important_words={"no","not","nor"}
stop_words = stop_words-important_words

# Tokenization + stopword removal

def tokenize_text(text):
  tokens=text.split()
  tokens=[word for word in tokens if word not in stop_words]
  return tokens

#Apply Preprocesing
df["tokens"]=df["clean_text"].apply(tokenize_text)

#Convert tokens back to text for TF-IDF
df["processed_text"] =df["tokens"].apply(lambda tokens:" ".join(tokens))

#Token Count for Validation
df["token_count"] = df["tokens"].apply(len)

#Validation
print("Total Documents:", len(df))
print("Empty Documents:", (df["token_count"] == 0).sum())

print("\nToken Count Statistics:")
print(df["token_count"].describe())

print("\nCleaned Text:")
print(df.loc[0, "clean_text"][:500])

print("\nTokens:")
print(df.loc[0, "tokens"][:30])

print("\nProcessed Text:")
print(df.loc[0, "processed_text"][:500])


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Total Documents: 50000
Empty Documents: 0

Token Count Statistics:
count    50000.000000
mean       102.030840
std         11.162579
min         72.000000
25%         94.000000
50%        101.000000
75%        109.000000
max        149.000000
Name: token_count, dtype: float64

Cleaned Text:
unable to access your login password step by step if you can no longer use your sign in password you can restore access yourself in a few steps go to the sign in page and select the option to reset your password enter the email address linked to your account so a recovery message can be sent choose a new password you have not used before mixing letters numbers and symbols confirm the new password and sign in again to make sure it works if you no longer have the linked email use another method su

Tokens:
['unable', 'access', 'login', 'password', 'step', 'step', 'no', 'longer', 'use', 'sign', 'password', 'restore', 'access', 'steps', 'go', 'sign', 'page', 'select', 'option', 'reset', 'password', 'ent

In [18]:
# ============================================================
# STEP 5: TF-IDF BASELINE MODEL + COSINE SIMILARITY SEARCH
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Crete Tf-IDF vectorizer

tfidf_vectorizer = TfidfVectorizer()

#Learn Vocabulary and convert docs into tf-idf vectors
tfidf_matrix=tfidf_vectorizer.fit_transform(df["processed_text"])

#Display TF-IDF information
print("TF_IDF Matrix Shape:",tfidf_matrix.shape)
print("Vocabulary Size:",len(tfidf_vectorizer.vocabulary_))

#Create TF-IDF Search Function

def search_tfidf(query,top_k=5):

  #Applying the same cleaning used for docs
  cleaned_query=clean_text(query)

  #Tokenize and Remove Stop Words
  query_tokens=tokenize_text(cleaned_query)

  #Convert tokens back to text for tf-idf
  processed_query=" ".join(query_tokens)

  #Convert user query into TF-IDF Vector
  query_vector=tfidf_vectorizer.transform([processed_query])

  #Calculate Cosine Similarity between query and all docs

  similarity_scores = cosine_similarity(query_vector,tfidf_matrix).flatten()

  #Get indices of documents with highest similarity
  top_indices = similarity_scores.argsort()[::-1][:top_k]

  #Retrieve document information
  results=df.loc[top_indices][["document_id","category","title","content"]].copy()

  #Add similarity score
  results["similarity_score"]=similarity_scores[top_indices]

  #Reset index for clean output
  results = results.reset_index(drop=True)

  #Start rank from 1
  results.index = results.index+1
  results.index.name = "Rank"

  return results

# ============================================================
# GET QUERY AND TOP-K VALUE FROM USER
# ============================================================

# Get query from user
query = input("Enter your search query: ").strip()

# Validate query
if not query:
    print("Please enter a valid search query.")

else:
    try:
        # Get K value from user
        top_k = int(input("Enter number of results to return (K): "))

        # Validate K
        if top_k <= 0:
            print("K must be greater than 0.")

        elif top_k > len(df):
            print(f"K cannot be greater than total documents ({len(df)}).")

        else:
            # Perform search
            results = search_tfidf(query, top_k)

            # Display results
            print("\n" + "=" * 70)
            print("TF-IDF SEARCH RESULTS")
            print("=" * 70)

            print("\nQuery:", query)
            print("Top K:", top_k)

            display(
                results[
                    [
                        "document_id",
                        "category",
                        "title",
                        "similarity_score"
                    ]
                ]
            )

    except ValueError:
        print("Please enter K as a whole number, for example: 5")

TF_IDF Matrix Shape: (50000, 736)
Vocabulary Size: 736
Enter your search query: I cannot get into my account anymore
Enter number of results to return (K): 5

TF-IDF SEARCH RESULTS

Query: I cannot get into my account anymore
Top K: 5


,document_id,category,title,similarity_score
Rank,,,,
1,KB005804,Account Management,Get Back Into Your Account in the App,0.190686
2,KB000353,Authentication,Get Back Into Your Account,0.163613
3,KB005116,Account Management,Steps to Recover Your Old Account Step by Step,0.152350
4,KB000432,Authentication,Get Back Into Your Account Quick Guide,0.145473
5,KB005564,Account Management,Steps to Recover Your Closed Account on iOS,0.144875


In [17]:
# ============================================================
# STEP 6: WORD2VEC SEMANTIC SEARCH MODEL
# ============================================================

from gensim.models  import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

#Training Word2vec model

word2vec_model = Word2Vec(sentences=df["tokens"],vector_size=100,window=5,min_count=2,workers=4,sg=1,epochs=10)

print("Word2vec training completed")
print("Vocabulary Size:",len(word2vec_model.wv))

#Creating function to convert tokens into one document vector

def get_word2vec_vector(tokens):

  #Keep only the words present in vocabulary
  valid_words=[word for word in tokens if word in word2vec_model.wv]

  #If no known words are available,return zero vector
  if len(valid_words) ==0:
    return np.zeros(word2vec_model.vector_size)

  #Get vectors for all valid words
  word_vectors=[word2vec_model.wv[word] for word in valid_words]

  # Average word vectors to create one document vector

  return np.mean(word_vectors,axis=0)


#convert every document into a Word2vec documnet vector

document_vectors_w2v=np.vstack(df["tokens"].apply(get_word2vec_vector))

print("Document Vector Matrix Shape:",document_vectors_w2v.shape)

# creating word2vec search function

def search_word2vec(query,top_k=5):

  #Applying Same preprocessing used for documents
  cleaned_query=clean_text(query)

  query_tokens=tokenize_text(cleaned_query)

  # Convert query tokens into Word2Vec vector
  query_vector = get_word2vec_vector(query_tokens)

  #check if query contains any usable vocabulary
  if np.all(query_vector==0):
    print("No known words were found in the query.")
    return None

  #Reshaping query to 2D for cosine similarity
  query_vector = query_vector.reshape(1,-1)

  #Calculate Cosine Similarity

  similarity_scores = cosine_similarity(query_vector,document_vectors_w2v).flatten()

  #Get Top-K document indices
  top_indices = similarity_scores.argsort()[::-1][:top_k]

  #Retrieve results
  results = df.iloc[top_indices][["document_id","category","title","content"]].copy()

  #Adding similarity score
  results["similarity_score"]=similarity_scores[top_indices]

  #Clean reanking Output
  results=results.reset_index(drop=True)
  results.index = results.index+1
  results.index.name="Rank"

  return results

#Getting query and K value from user

query=input("Enter Your Search query:").strip()

top_k = int(input("Enter the no.of results to return (k):"))

#Performing Search
results_w2v=search_word2vec(query,top_k)

#Display results

if results_w2v is not None:

  print("\n" + "=" * 70)
  print("WORD2VEC SEARCH RESULTS")
  print("=" * 70)

  print("\nQuery:",query)
  print("Top k:",top_k)

  display(
      results_w2v[
          [
              "document_id",
              "category",
              "title",
              "similarity_score"
          ]
      ]
  )

Word2vec training completed
Vocabulary Size: 736
Document Vector Matrix Shape: (50000, 100)
Enter Your Search query:I cannot get into my account anymore
Enter the no.of results to return (k):5

WORD2VEC SEARCH RESULTS

Query: I cannot get into my account anymore
Top k: 5


,document_id,category,title,similarity_score
Rank,,,,
1,KB035236,Authentication,Recover a Lost Locked-Out Account on Mobile,0.665610
2,KB021379,Security,Resetting a Forgotten Hacked Account on iOS,0.662211
3,KB002086,Authentication,Regain Access to Your Blocked Account on iOS,0.658542
4,KB021499,Security,What to Do If You Lose Your Stolen Account in ...,0.657024
5,KB043502,Security,How to Reset Your Stolen Account Explained,0.654699


In [21]:
# ============================================================
# STEP 7: FASTTEXT SEMANTIC SEARCH MODEL
# ============================================================

from gensim.models import FastText
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

#Train FastText Model

fasttext_model = FastText(sentences=df["tokens"],vector_size=100,window=5,min_count=2,workers=4,sg=1,epochs=10,min_n=3,max_n=6)

print("FastText Training Completed.")
print("Vocabulary Size",len(fasttext_model.wv))
print("Vector Size:",fasttext_model.vector_size)

#Convert Tokens into one fasttext vector

def get_fasttext_vector(tokens):

  if len(tokens)==0:
    return np.zeros(fasttext_model.vector_size)

  word_vectors=[fasttext_model.wv[word] for word in tokens]

  return np.mean(word_vectors,axis=0)

#Convert every document into a fasttext document vector

document_vectors_ft = np.vstack(df["tokens"].apply(get_fasttext_vector))
print("Document Vector Matrix Shape:",document_vectors_ft.shape)

# Create FastText Search function

def search_fasttext(query,top_k=5):

  #Apply same preprocesing used for documents
  cleaned_query=clean_text(query)

  query_tokens = tokenize_text(cleaned_query)

  #Convert query into FastText Vector
  query_vector=get_fasttext_vector(query_tokens)

  # Check for empty query after preprocessing
  if np.all(query_vector==0):
    print("No usable words were found in the query.")
    return None

  #Convert shape from (100,) to (1,100):
  query_vector=query_vector.reshape(1,-1)

  #Calculating cosine similarity with all documents
  similarity_scores=cosine_similarity(query_vector,document_vectors_ft).flatten()

  #Get Top-K document indices
  top_indices = similarity_scores.argsort()[::-1][:top_k]

  #Retrieve Matching documents
  results = df.iloc[top_indices][
      [
          "document_id",
          "category",
          "title",
          "content"
      ]
  ].copy()

  #Add Similarity Scores

  results["similarity_score"] = similarity_scores[top_indices]

  #Create clean ranking
  results=results.reset_index(drop=True)
  results.index = results.index+1
  results.index.name ="Rank"

  return results

# Get query and k value from user

query=input("Enter your search query:").strip()

try:
  top_k = int(input("Enter No.of results to return(k)"))

  if not query:
    print("Please enter a valid search query.")

  elif top_k <=0:
    print("K must be greater than 0.")

  elif top_k > len(df):
    print(f"K cannot be greater than total documents ({len(df)}).")

  else:

    #performing search
    results_ft=search_fasttext(query,top_k)

    #displaying results

    if results_ft is not None:
      print("\n" + "=" *70)
      print("FASTTEXT SEARCH RESULTS")
      print("=" *70)

      print("\nQuery:",query)
      print("Top_K:",top_k)

      display(results_ft[
          [
              "document_id",
              "category",
              "title",
              "similarity_score"
          ]
      ])

except ValueError:
  print("Please enter K as a whole number, for example 5")

FastText Training Completed.
Vocabulary Size 736
Vector Size: 100
Document Vector Matrix Shape: (50000, 100)
Enter your search query:I forgot my password and cannot login
Enter No.of results to return(k)5

FASTTEXT SEARCH RESULTS

Query: I forgot my password and cannot login
Top_K: 5


,document_id,category,title,similarity_score
Rank,,,,
1,KB035567,Authentication,Forgot Your Password? on the Web,0.773029
2,KB035183,Authentication,How to Reset Your Password on Android,0.766095
3,KB002161,Authentication,Steps to Recover Your Password on Mobile,0.765893
4,KB000065,Authentication,Fix Login Password Access Problems FAQ,0.763896
5,KB003057,Authentication,What to Do If You Lose Your Login Password for...,0.763181


In [26]:
# ============================================================
# STEP 8: MODEL COMPARISON
# TF-IDF vs Word2Vec vs FastText
# ============================================================

def compare_models(query, top_k=5):

    print("=" * 80)
    print("MODEL COMPARISON")
    print("=" * 80)

    print("Query:", query)
    print("Top K:", top_k)

    # --------------------------------------------------------
    # 1. TF-IDF Results
    # --------------------------------------------------------

    tfidf_results = search_tfidf(query, top_k)

    print("\n" + "=" * 80)
    print("TF-IDF RESULTS")
    print("=" * 80)

    display(
        tfidf_results[
            [
                "document_id",
                "category",
                "title",
                "similarity_score"
            ]
        ]
    )

    # --------------------------------------------------------
    # 2. Word2Vec Results
    # --------------------------------------------------------

    word2vec_results = search_word2vec(query, top_k)

    print("\n" + "=" * 80)
    print("WORD2VEC RESULTS")
    print("=" * 80)

    if word2vec_results is not None:
        display(
            word2vec_results[
                [
                    "document_id",
                    "category",
                    "title",
                    "similarity_score"
                ]
            ]
        )

    # --------------------------------------------------------
    # 3. FastText Results
    # --------------------------------------------------------

    fasttext_results = search_fasttext(query, top_k)

    print("\n" + "=" * 80)
    print("FASTTEXT RESULTS")
    print("=" * 80)

    if fasttext_results is not None:
        display(
            fasttext_results[
                [
                    "document_id",
                    "category",
                    "title",
                    "similarity_score"
                ]
            ]
        )

    return tfidf_results, word2vec_results, fasttext_results


# ============================================================
# USER INPUT
# ============================================================

query = input("Enter your search query: ").strip()

try:
    top_k = int(input("Enter number of results to return (K): "))

    if not query:
        print("Please enter a valid search query.")

    elif top_k <= 0:
        print("K must be greater than 0.")

    elif top_k > len(df):
        print(f"K cannot be greater than total documents ({len(df)}).")

    else:
        tfidf_results, word2vec_results, fasttext_results = compare_models(
            query,
            top_k
        )

except ValueError:
    print("Please enter K as a whole number, for example: 5")

Enter your search query: How can I regain access to my profile?
Enter number of results to return (K): 5
MODEL COMPARISON
Query: How can I regain access to my profile?
Top K: 5

TF-IDF RESULTS


,document_id,category,title,similarity_score
Rank,,,,
1,KB003288,Account Management,End Your Profile FAQ,0.379349
2,KB036759,Account Management,End Your Profile in the App,0.365607
3,KB003736,Account Management,Review Your Profile in the App,0.363520
4,KB005928,Account Management,Manage Your Profile FAQ,0.363001
5,KB003304,Account Management,Remove a Membership on iOS,0.357034



WORD2VEC RESULTS


,document_id,category,title,similarity_score
Rank,,,,
1,KB003750,Account Management,Restore Access to Your Deleted Data on Android,0.698644
2,KB044820,Subscriptions,Fix Ended Membership Access Problems Quick Guide,0.696996
3,KB044688,Subscriptions,Recover Your Ended Membership Explained,0.695516
4,KB023836,Subscriptions,Reset Your Expired Subscription for New Users,0.692566
5,KB029390,Website,Resetting a Forgotten Site Preferences,0.687714



FASTTEXT RESULTS


,document_id,category,title,similarity_score
Rank,,,,
1,KB044820,Subscriptions,Fix Ended Membership Access Problems Quick Guide,0.722888
2,KB029445,Website,Saved Site Settings Recovery Guide for Busines...,0.720183
3,KB003750,Account Management,Restore Access to Your Deleted Data on Android,0.718980
4,KB044040,Subscriptions,Restore Access to Your Expired Subscription St...,0.718387
5,KB023836,Subscriptions,Reset Your Expired Subscription for New Users,0.717862


In [27]:
# ============================================================
# STEP 9: QUANTITATIVE EVALUATION USING PRECISION@K
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# 1. Create a small evaluation set
# ------------------------------------------------------------

evaluation_queries = [
    {
        "query": "I forgot my password and cannot login",
        "expected_category": "Authentication"
    },
    {
        "query": "I want to change my account details",
        "expected_category": "Account Management"
    },
    {
        "query": "My payment did not go through",
        "expected_category": "Payments"
    },
    {
        "query": "Where is my order",
        "expected_category": "Orders"
    },
    {
        "query": "When will my package arrive",
        "expected_category": "Shipping"
    },
    {
        "query": "I want my money back",
        "expected_category": "Refunds"
    },
    {
        "query": "My subscription has expired",
        "expected_category": "Subscriptions"
    },
    {
        "query": "The mobile app is not working",
        "expected_category": "Mobile App"
    },
    {
        "query": "I am not receiving notifications",
        "expected_category": "Notifications"
    },
    {
        "query": "I think my account has been hacked",
        "expected_category": "Security"
    },
    {
        "query": "The website is not loading",
        "expected_category": "Website"
    },
    {
        "query": "I need help from customer service",
        "expected_category": "Customer Support"
    }
]


# ------------------------------------------------------------
# 2. Precision@K function
# ------------------------------------------------------------

def precision_at_k(results, expected_category, k):

    if results is None or len(results) == 0:
        return 0.0

    top_results = results.head(k)

    relevant_count = (
        top_results["category"] == expected_category
    ).sum()

    precision = relevant_count / k

    return precision


# ------------------------------------------------------------
# 3. Evaluate all three models
# ------------------------------------------------------------

def evaluate_models(evaluation_queries, k=5):

    evaluation_results = []

    for item in evaluation_queries:

        query = item["query"]
        expected_category = item["expected_category"]

        # Run all three models
        tfidf_results = search_tfidf(query, k)
        word2vec_results = search_word2vec(query, k)
        fasttext_results = search_fasttext(query, k)

        # Calculate Precision@K
        tfidf_precision = precision_at_k(
            tfidf_results,
            expected_category,
            k
        )

        word2vec_precision = precision_at_k(
            word2vec_results,
            expected_category,
            k
        )

        fasttext_precision = precision_at_k(
            fasttext_results,
            expected_category,
            k
        )

        evaluation_results.append({
            "Query": query,
            "Expected Category": expected_category,
            "TF-IDF Precision@K": tfidf_precision,
            "Word2Vec Precision@K": word2vec_precision,
            "FastText Precision@K": fasttext_precision
        })

    return pd.DataFrame(evaluation_results)


# ------------------------------------------------------------
# 4. Run evaluation
# ------------------------------------------------------------

K = 5

evaluation_df = evaluate_models(
    evaluation_queries,
    k=K
)


# ------------------------------------------------------------
# 5. Display individual query results
# ------------------------------------------------------------

print("=" * 100)
print(f"MODEL EVALUATION - PRECISION@{K}")
print("=" * 100)

display(evaluation_df)


# ------------------------------------------------------------
# 6. Calculate average Precision@K
# ------------------------------------------------------------

average_scores = pd.DataFrame({
    "Model": [
        "TF-IDF",
        "Word2Vec",
        "FastText"
    ],
    f"Average Precision@{K}": [
        evaluation_df["TF-IDF Precision@K"].mean(),
        evaluation_df["Word2Vec Precision@K"].mean(),
        evaluation_df["FastText Precision@K"].mean()
    ]
})


print("\n" + "=" * 60)
print("AVERAGE MODEL PERFORMANCE")
print("=" * 60)

display(average_scores)

MODEL EVALUATION - PRECISION@5


,Query,Expected Category,TF-IDF Precision@K,Word2Vec Precision@K,FastText Precision@K
0,I forgot my password and cannot login,Authentication,1.0,1.0,1.0
1,I want to change my account details,Account Management,1.0,1.0,1.0
2,My payment did not go through,Payments,1.0,1.0,1.0
3,Where is my order,Orders,1.0,1.0,1.0
4,When will my package arrive,Shipping,1.0,1.0,1.0
5,I want my money back,Refunds,1.0,1.0,0.6
6,My subscription has expired,Subscriptions,1.0,1.0,1.0
7,The mobile app is not working,Mobile App,1.0,0.8,0.8
8,I am not receiving notifications,Notifications,1.0,1.0,0.6
9,I think my account has been hacked,Security,1.0,1.0,1.0



AVERAGE MODEL PERFORMANCE


,Model,Average Precision@5
0,TF-IDF,0.916667
1,Word2Vec,0.900000
2,FastText,0.833333


**

> **Overall, the results show that TF-IDF performs strongly on clean, keyword-aligned queries, while embedding-based approaches provide broader semantic matching, with FastText offering an additional advantage in handling unseen and misspelled words through subword representations.**
**